# Week 3: Recommendation Systems

<small>OPAN 6604. Dataset: `bookrec/data` (`Books.csv`, `Ratings.csv`). Goal: predict the rating a user would give a book they have not rated, then turn those predictions into top-N recommendations.</small>

<small>This is the in-class demo: **collaborative filtering** with the [`surprise`](http://surpriselib.com/) library, user-based CF (UBCF), item-based CF (IBCF), and a proper hold-out evaluation. </small>

<small>**Note on similarity:** both CF flavors rest on a similarity measure between rows (users) or columns (movies) of the rating matrix. We use cosine and Pearson - the two covered in lecture.</small>

---
## Step 0: Setup

<small>Install `scikit-surprise` if needed (pick the path for your environment in the cell below), then import the CF building blocks.</small>

In [1]:
# Install scikit-surprise if it's not already available.
#   All platforms (Colab / Windows / macOS / Linux):
# Note: `numpy<2.0` is required because scikit-surprise's compiled extensions
#   were built against the NumPy 1.x and break under NumPy 2.x.

#       %pip install "numpy<2.0"
#       %pip install scikit-surprise

#   Fallback (only if pip can't find a wheel for your Python and tries to
#   compile, e.g. "Microsoft Visual C++ required") -> use conda-forge:
#       conda install -c conda-forge scikit-surprise -y


In [2]:
# Surprise's CF building blocks:
#   KNNBasic        — neighborhood CF (UBCF or IBCF, toggled via sim_options)
#   BaselineOnly    — global/user/item-mean baseline, useful for comparison
#   Dataset/Reader  — wrap a pandas DataFrame as a Surprise dataset
#   accuracy        — RMSE, MAE, etc. on prediction lists
#   RandomizedSearchCV / GridSearchCV — Surprise's built-in CV hyperparameter search
from pathlib import Path

import pandas as pd
import numpy as np
from collections import defaultdict
from surprise import KNNBasic, BaselineOnly, Dataset, Reader, accuracy
from surprise.model_selection import (
    train_test_split, KFold, RandomizedSearchCV, GridSearchCV,
)

---
## Step 1: Load data

<small>Load the catalog and the ratings, and build a `movieId` → title lookup for displaying recommendations later. Each `movieId` is unique, so we key everything off it.</small>

In [3]:
# Load the book catalog and ratings from bookrec/data.
def find_data_dir() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        data_dir = base / "bookrec" / "data"
        if data_dir.exists():
            return data_dir
    raise FileNotFoundError("Could not find bookrec/data from the current working directory")

DATA_DIR = find_data_dir()
books = pd.read_csv(DATA_DIR / "Books.csv")
ratings = pd.read_csv(DATA_DIR / "Ratings.csv")

# Optional cleanup: keep only the columns we need and drop malformed rows.
books = books.dropna(subset=["book_id", "title"])
ratings = ratings.dropna(subset=["user_id", "book_id", "rating"])

books["book_id"] = books["book_id"].astype(int)
ratings["book_id"] = ratings["book_id"].astype(int)
ratings["user_id"] = ratings["user_id"].astype(int)
ratings["rating"] = ratings["rating"].astype(float)

# Helpful display lookup.
title_of = dict(zip(books["book_id"], books["title"]))
print(f"Books: {books.shape}  |  Ratings: {ratings.shape}")

# Optional display with author.
label_of = {
    row.book_id: f"{row.title} — {row.authors}"
    for row in books[["book_id", "title", "authors"]].itertuples(index=False)
}

Books: (9964, 16)  |  Ratings: (164728, 3)


In [4]:
# Wrap the long-format ratings into a Surprise dataset.
# MovieLens uses a 0.5–5.0 half-star scale. Using Reader we clip predictions to this range.
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(
    ratings[["user_id", "book_id", "rating"]],
    reader
)

---
## Step 2: Train/Test Split (held-out evaluation)

<small>Carve out a 90/10 hold-out split **up front**, before any modeling or tuning. The 10% test set is touched only once, at the very end, to score the final models. All hyperparameter tuning (below) uses cross-validation *within* the 90% training set, so the test set stays a true measure of generalization. `random_state` keeps the split reproducible.</small>

In [5]:
# Build a 90/10 train/test split FIRST so the held-out 10% is never seen during tuning.
# All cross-validation for hyperparameter search happens within `trainset` only.
# `random_state` makes the split reproducible across runs.
trainset, testset = train_test_split(data, test_size=0.1, random_state=6604)
print(f"Train: {trainset.n_ratings} ratings  |  Test: {len(testset)} ratings")

# Also keep a full-data trainset purely for the illustrative top-N demo in Part 1.
full_trainset = data.build_full_trainset()
print(f"Full trainset (demo only): {full_trainset.n_users} users, "
      f"{full_trainset.n_items} movies, {full_trainset.n_ratings} ratings")

Train: 148255 ratings  |  Test: 16473 ratings
Full trainset (demo only): 1192 users, 9229 movies, 164728 ratings


---
## Step 3: Baseline Collaborative Filtering

### Precision and Recall @ K

<small>Frame recommendation as classification: a movie is "relevant" if its true rating ≥ a threshold (4.0 here). For each user we rank predictions by estimated rating, then ask — of the top-N we'd show, how many were relevant (**precision**), and of all their relevant movies, how many made the top-N (**recall**). We average across users.</small>

In [6]:
# Top-N precision and recall for ranking quality:
#   - a movie is "relevant" if its true rating is >= threshold (4.0 here),
#   - for each user we rank predictions by estimated rating,
#   - precision = fraction of the user's top-N that were actually relevant,
#   - recall    = fraction of the user's relevant items captured in the top-N.
# `top_n` is the list length - distinct from the model's k (the neighborhood size).
# Returns the mean across all users in the predictions list.
def precision_recall_at_k(predictions, top_n=10, threshold=4.0):
    user_data = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_data[uid].append((est, true_r))
    precisions, recalls = [], []
    for items in user_data.values():
        items.sort(key=lambda x: x[0], reverse=True)
        n_relevant = sum(1 for _, t in items if t >= threshold)
        n_hits = sum(1 for _, t in items[:top_n] if t >= threshold)
        precisions.append(n_hits / top_n)
        if n_relevant > 0:
            recalls.append(n_hits / n_relevant)
    return np.mean(precisions), np.mean(recalls)

#### Top-N Recommendations

<small>CF tends to surface rarely-rated movies. Restricting candidates to movies with at least `MIN_RATINGS` ratings is a simple, effective fix. The helper below is reused for both models.</small>

In [7]:
# Recommend a user's top-N unseen movies, ranked by predicted rating.
# Popularity filter (>= MIN_RATINGS): drop movies with very few ratings, which CF
# could otherwise rank highly on almost no evidence (e.g. a single 5/5 rating).
# MIN_RATINGS=20 is just a suggested value - the right cutoff depends on the
# dataset size, catalog, and use case.
MIN_RATINGS = 20
counts = ratings["book_id"].value_counts()
popular_books = set(counts[counts >= MIN_RATINGS].index)
def top_n_for_user(model, user_id, top_n=5):
    seen = set(ratings.loc[ratings["user_id"] == user_id, "book_id"])
    scored = []
    for book_id in books["book_id"]:
        if book_id in seen or book_id not in popular_books:
            continue
        est = model.predict(user_id, book_id).est
        scored.append((title_of.get(book_id, f"Book {book_id}"), est))
    return sorted(scored, key=lambda x: -x[1])[:top_n]

### User-Based CF

<small>Similarity is computed between **users** (rows): each prediction averages the `k` most similar users who rated the target movie - "people like you also enjoyed...". We use **Pearson**, which centers each user on their own mean - important because users rate on different scales (some skew high, some low).</small>

In [8]:
# User-based: the k=10 most similar users (Pearson similarity) vote on each prediction.
ubcf = KNNBasic(k=10, sim_options={"name": "pearson", "user_based": True}, verbose=False)
ubcf.fit(full_trainset)

# Print precision and recall on the held-out testset
ubcf_predictions = ubcf.test(testset)
ubcf_precision, ubcf_recall = precision_recall_at_k(ubcf_predictions, top_n=10, threshold=4.0)
print(f"UBCF precision@10: {ubcf_precision:.4f}  |  recall@10: {ubcf_recall:.4f}")

UBCF precision@10: 0.7479  |  recall@10: 0.8948


In [9]:
top_n_for_user(ubcf, user_id=1, top_n=5)   # top-5 UBCF recommendations for user 1

[('The Hunger Games (The Hunger Games, #1)', 3.8404946335777765),
 ("Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",
  3.8404946335777765),
 ('Twilight (Twilight, #1)', 3.8404946335777765),
 ('To Kill a Mockingbird', 3.8404946335777765),
 ('The Great Gatsby', 3.8404946335777765)]

### Item-Based CF

<small>Now similarity is between **movies** (columns), from how the same users rated them - "because you liked X, you might like Y...". We also switch to **cosine**, the common metric for item-based CF. So *both* knobs change from UBCF - the axis (users→movies) and the similarity - which mirrors how the metric is usually chosen per method in practice. Compare this top-5 to the UBCF list: same user, different lens, different movies.</small>

In [10]:
# Item-based: similarity between movies, using cosine (the common item-based choice).
ibcf = KNNBasic(k=10, sim_options={"name": "cosine", "user_based": False}, verbose=False)
ibcf.fit(full_trainset)

# Print precision and recall on the held-out testset
ibcf_predictions = ibcf.test(testset)
ibcf_precision, ibcf_recall = precision_recall_at_k(ibcf_predictions, top_n=10, threshold=4.0)
print(f"IBCF precision@10: {ibcf_precision:.4f}  |  recall@10: {ibcf_recall:.4f}")

IBCF precision@10: 0.6792  |  recall@10: 0.8184


#### Top-N Recommendations

In [11]:
# Top-5 IBCF recommendations for user 1 (same helper, different model).
top_n_for_user(ibcf, user_id=1, top_n=5)

[('The Hunger Games (The Hunger Games, #1)', 3.8404946335777765),
 ("Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",
  3.8404946335777765),
 ('Twilight (Twilight, #1)', 3.8404946335777765),
 ('To Kill a Mockingbird', 3.8404946335777765),
 ('The Great Gatsby', 3.8404946335777765)]

---
## Step 4: Tuning

<small>CF is trained without labeled "correct answers," yet we evaluate it like supervised learning: hold out known ratings, predict them, and compare. We report a **rating-accuracy** metric (RMSE) and **ranking** metrics (Precision@K, Recall@K)

### f1_score, f1_at_k

In [12]:
# --- Make Surprise's search optimize F1@10 ---------------------------------
# Surprise's CV search only ranks built-in accuracy measures. We register an
# F1@10 scorer in the `accuracy` module and bind it to the `fcp` name, which is
# the framework's single higher-is-better ranking slot (argmax) — a match for F1
# (we don't use the real FCP metric anywhere else). cv_results will store it under
# 'fcp'; we relabel it to 'F1@10' in all user-facing output.
TOP_N = 10            # list length for precision/recall@k
THRESHOLD = 4.0       # rating >= threshold counts as "relevant"
CV_FOLDS = 3          # folds for cross-validation during tuning

def f1_score(precision, recall):
    # Harmonic mean of precision and recall; 0 when both are 0.
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def f1_at_k(predictions, verbose=0):
    # accuracy-module-style scorer: takes a list of Surprise predictions,
    # returns the F1 of Precision@TOP_N and Recall@TOP_N (a single float).
    p, r = precision_recall_at_k(predictions, top_n=TOP_N, threshold=THRESHOLD)
    return f1_score(p, r)

def recall_at_k(predictions, verbose=0):
    _, r = precision_recall_at_k(predictions, top_n=TOP_N, threshold=THRESHOLD)
    return r

accuracy.fcp = recall_at_k   # route Recall@10 through the higher-is-better ranking slot
MEASURES = ["fcp", "rmse"]   # 'fcp' == our Recall@10 (ranked); 'rmse' reported alongside

### Tune `k` per model with Surprise's `RandomizedSearchCV` → `GridSearchCV`

<small>Same two-stage strategy as before, but the search loops are now Surprise's own **`RandomizedSearchCV`** and **`GridSearchCV`** (from `surprise.model_selection`), which handle the cross-validation internally. **Stage 1 — random search:** sample `k` values across a wide range. **Stage 2 — grid search:** exhaustively scan a narrow window around the random winner. Both run k-fold CV on the **training set only** (the test set is untouched).</small>

<small>Surprise's search ranks candidates by a measure in its `accuracy` module (`rmse`, `mae`, `mse`, `fcp`) — it has no built-in Precision/Recall/F1@K. To keep the **F1@10 objective** from the earlier notebooks, we register a custom F1@10 function with the `accuracy` module and route it through Surprise's only higher-is-better ranking slot (`fcp`), so the built-in search selects `k` by F1. RMSE is requested alongside it and reported per candidate. Per-candidate Precision@10/Recall@10 aren't separately exposed by the framework, but all four metrics are reported on the held-out test set at the end.</small>

In [13]:
# Two-stage tuning per model, using Surprise's built-in search frameworks.
#   Stage 1: RandomizedSearchCV samples N_RANDOM distinct k's from [K_MIN, K_MAX].
#   Stage 2: GridSearchCV scans every integer k in a +/- GRID_RADIUS window around
#            the random winner. Both rank by F1@10 (the 'fcp' slot) via CV.
# ONLY k is searched — the similarity settings are fixed per model, not tuned.
K_MIN, K_MAX = 5, 100       # overall search range for k
N_RANDOM     = 30          # random-search samples per model
GRID_RADIUS  = 5           # grid-search half-width around the random winner
RANDOM_SEED  = 6604

# Surprise's search frameworks split internally and require a Dataset, not a
# Trainset. Rebuild a Dataset from ONLY the 90% training ratings so the held-out
# test set is still never seen during tuning. trainset stores ratings as inner
# ids, so map them back to raw (user_id, book_id, rating) first.
train_df = pd.DataFrame(
    [(trainset.to_raw_uid(u), trainset.to_raw_iid(i), r)
     for (u, i, r) in trainset.all_ratings()],
    columns=["user_id", "book_id", "rating"],
)
train_data = Dataset.load_from_df(train_df[["user_id", "book_id", "rating"]], reader)

# Fixed (NOT tuned) similarity settings, one config per model.
sim_configs = {
    "UBCF pearson": {"name": "pearson", "user_based": True},
    "IBCF cosine":  {"name": "cosine",  "user_based": False},
}

def cv_results_df(search, stage):
    # Tidy a Surprise search's cv_results into rows, relabeling fcp -> F1@10.
    res = search.cv_results
    return pd.DataFrame({
        "stage": stage,
        "k": res["param_k"],
        "Recall@10": res["mean_test_fcp"],
        "RMSE": res["mean_test_rmse"],
    })

search_log, best_k = {}, {}
for name, sim in sim_configs.items():
    cv = KFold(n_splits=CV_FOLDS, random_state=RANDOM_SEED, shuffle=True)
    # Bake the fixed similarity config into the estimator so the search varies only k.
    # Surprise calls algo_class(**params); this wrapper supplies the fixed args.
    def make_knn(sim=sim):
        return lambda **params: KNNBasic(sim_options=sim, verbose=False, **params)
    knn = make_knn()

    # --- Stage 1: random search (k only) ---
    rs = RandomizedSearchCV(
        knn,
        param_distributions={"k": list(range(K_MIN, K_MAX + 1))},
        n_iter=N_RANDOM, measures=MEASURES, cv=cv,
        random_state=RANDOM_SEED, refit=False,
    )
    rs.fit(train_data)
    rand_k = rs.best_params["fcp"]["k"]   # 'fcp' == Recall@10

    # --- Stage 2: grid search around the random winner (k only) ---
    lo = max(K_MIN, rand_k - GRID_RADIUS)
    hi = min(K_MAX, rand_k + GRID_RADIUS)
    gs = GridSearchCV(
        knn,
        param_grid={"k": list(range(lo, hi + 1))},
        measures=MEASURES, cv=cv, refit=False,
    )
    gs.fit(train_data)

    # Best k over BOTH stages by Recall@10.
    combined = pd.concat([cv_results_df(rs, "random"), cv_results_df(gs, "grid")],
                         ignore_index=True)
    winner = combined.loc[combined["Recall@10"].idxmax()]
    best_k[name] = int(winner["k"])
    search_log[name] = combined

    print(f"{name}: random winner k={rand_k}  -> grid scanned {lo}..{hi}  "
          f"-> best k={best_k[name]}  "
          f"Recall@10={winner['Recall@10']:.4f}  RMSE={winner['RMSE']:.4f}")

UBCF pearson: random winner k=19  -> grid scanned 14..24  -> best k=21  Recall@10=0.2978  RMSE=1.0648
IBCF cosine: random winner k=71  -> grid scanned 66..76  -> best k=71  Recall@10=0.2860  RMSE=0.8681


In [14]:
# Per-candidate detail for each model: which stage (random vs grid) each k came
# from, its CV F1@10 (the ranking objective) and CV RMSE. Pulled straight from
# Surprise's cv_results.
search_df = pd.concat(
    [df.assign(Model=name) for name, df in search_log.items()],
    ignore_index=True,
)[["Model", "stage", "k", "Recall@10", "RMSE"]]
search_df.sort_values(["Model", "Recall@10"], ascending=[True, False]).round(4)

,Model,stage,k,Recall@10,RMSE
68,IBCF cosine,random,71,0.2860,0.8681
76,IBCF cosine,grid,71,0.2860,0.8681
79,IBCF cosine,grid,74,0.2859,0.8680
48,IBCF cosine,random,73,0.2859,0.8680
78,IBCF cosine,grid,73,0.2859,0.8680
...,...,...,...,...,...
1,UBCF pearson,random,11,0.2974,1.0681
34,UBCF pearson,grid,18,0.2974,1.0652
4,UBCF pearson,random,9,0.2973,1.0705
19,UBCF pearson,random,7,0.2965,1.0754


---
## Step 5: Final Evaluation on the Held-Out Test Set

<small>Now — and only now — we touch the 10% test set. We fit each model on the full 90% training split using the **tuned `k`** found by Optuna (the baseline has no `k` to tune), then score all three on the held-out ratings. We report RMSE (rating accuracy) alongside Precision@10, Recall@10, and their F1 (ranking quality). The baseline is the bar CF must clear; note that a model's RMSE and its ranking quality don't always agree.</small>

In [15]:
# Fit each model on the FULL 90% training split, then score once on the held-out test set.
# UBCF/IBCF use the k tuned by Optuna; BaselineOnly has no k to tune and is included as the
# comparison bar. This is the only place the test set is used.
models = {
    "Baseline": BaselineOnly(verbose=False),
    "UBCF pearson": KNNBasic(k=best_k["UBCF pearson"],
                             sim_options=sim_configs["UBCF pearson"], verbose=False),
    "IBCF cosine": KNNBasic(k=best_k["IBCF cosine"],
                            sim_options=sim_configs["IBCF cosine"], verbose=False),
}

results = []
for name, m in models.items():
    m.fit(trainset)
    preds = m.test(testset)
    p, r = precision_recall_at_k(preds, top_n=TOP_N, threshold=THRESHOLD)
    results.append({
        "Model": name,
        "k": best_k.get(name, "—"),
        "RMSE": accuracy.rmse(preds, verbose=False),
        f"Precision@{TOP_N}": p,
        f"Recall@{TOP_N}": r,
        "F1": f1_score(p, r),
    })

pd.DataFrame(results).round(4)

,Model,k,RMSE,Precision@10,Recall@10,F1
0,Baseline,—,0.8423,0.6568,0.7912,0.7178
1,UBCF pearson,21,1.0287,0.6586,0.7930,0.7196
2,IBCF cosine,71,0.8565,0.6414,0.7734,0.7012


##### Evaluate Baseline Collaborative Filtering Models from Step 3 on Held-Out Test Set

In [16]:
# User-based: the k=10 most similar users (Pearson similarity) vote on each prediction.
ubcf = KNNBasic(k=10, sim_options={"name": "pearson", "user_based": True}, verbose=False)
ubcf.fit(full_trainset)

# Print precision and recall on the held-out testset
ubcf_predictions = ubcf.test(testset)
ubcf_precision, ubcf_recall = precision_recall_at_k(ubcf_predictions, top_n=10, threshold=4.0)
print(f"UBCF precision@10: {ubcf_precision:.4f}  |  recall@10: {ubcf_recall:.4f}, f1: {f1_score(ubcf_precision, ubcf_recall):.4f}")

UBCF precision@10: 0.7479  |  recall@10: 0.8948, f1: 0.8148


In [17]:
# Item-based: similarity between movies, using cosine (the common item-based choice).
ibcf = KNNBasic(k=10, sim_options={"name": "cosine", "user_based": False}, verbose=False)
ibcf.fit(full_trainset)

# Print precision and recall on the held-out testset
ibcf_predictions = ibcf.test(testset)
ibcf_precision, ibcf_recall = precision_recall_at_k(ibcf_predictions, top_n=10, threshold=4.0)
print(f"IBCF precision@10: {ibcf_precision:.4f}  |  recall@10: {ibcf_recall:.4f}, f1: {f1_score(ibcf_precision, ibcf_recall):.4f}")

IBCF precision@10: 0.6792  |  recall@10: 0.8184, f1: 0.7423
